<a id='s18'></a>
---
# __ SECTION 18 -- Career Roadmap, 2030-Ready Projects & Resources

<div style="background: linear-gradient(135deg, #0f0c29, #302b63); padding: 20px 25px; border-radius: 12px; color: white;">
<h3 style="margin:0; font-size:1.3em;">_ This section builds the portfolio that gets you hired in 2030</h3>
<p style="margin:8px 0 0 0; color:#a8edea;">Not what everyone else has done -- but projects that showcase frontier thinking.</p>
</div>

---

## 18.1 Why "Standard" Projects Will Fail You by 2030

The classic portfolio checklist -- MNIST classifier, face detection, dog vs cat -- will be dead weight by 2030:

- Basic YOLO demos: **every bootcamp graduate has one**
- Transfer learning on CIFAR: **automated in 10 lines with AutoML today**
- Face attendance system: **solved, commoditized, and ethically challenged**

What 2030 employers actually want:

| 2024 (Saturated) | 2030 (High Signal) |
|------------------|--------------------|
| "I fine-tuned ResNet" | Multimodal reasoning pipelines |
| "I ran YOLO on webcam" | Neuromorphic / efficient edge CV |
| "I trained on MNIST" | Synthetic data generation + curriculum learning |
| "Face recognition" | Privacy-preserving CV with federated learning |
| "Image classifier" | Vision-Language-Action models |

---

## 18.2 The 2030-Ready Portfolio Projects

Build **3 of these 8 projects** to stand in the top 1% of CV portfolios.

---

### _ PROJECT A -- Multimodal Visual Question Answering Engine
**Difficulty:** _____ | **Impact:** Maximum  
**Why it's impressive:** Combines vision + language at the architecture level -- the exact intersection that every major AI lab is racing toward.

**What you build:** A system that accepts an image + natural language question and produces a grounded, accurate answer -- with the spatial region highlighted that justifies the answer.

**Stack:** CLIP (vision-language encoder) + GPT-style decoder + Grad-CAM grounding

---

### _ PROJECT B -- Self-Supervised Visual Representation Learning
**Difficulty:** _____ | **Impact:** Research-grade  
**Why it's impressive:** You understand the frontier: models that learn without any labels (DINO, MAE, SimCLR). This is what Meta AI, Google Brain, and DeepMind publish papers on.

**What you build:** A DINO-style self-supervised trainer that learns visual features on unlabeled images, then demonstrates downstream performance on classification with zero labels.

---

### _ PROJECT C -- Neural Radiance Field (NeRF) Scene Reconstruction
**Difficulty:** _____ | **Impact:** 3D Vision frontier  
**Why it's impressive:** NeRF/3D Gaussian Splatting is the backbone of spatial computing (Apple Vision Pro, Meta AR glasses). Almost no junior candidates understand it.

**What you build:** A mini-NeRF that takes 20 photos of an object and synthesizes novel view angles.

---

### _ PROJECT D -- Privacy-Preserving CV with Federated Learning
**Difficulty:** ____ | **Impact:** Ethics + Engineering  
**Why it's impressive:** Shows you think beyond accuracy -- about privacy, regulation (GDPR, AI Act), and distributed systems. Healthcare and finance companies love this.

**What you build:** A federated image classification system where multiple "hospitals" train locally on sensitive medical images, only sharing model updates -- never raw data.

---

### _ PROJECT E -- Video Temporal Reasoning & Action Anticipation
**Difficulty:** ____ | **Impact:** Autonomous systems  
**Why it's impressive:** Most CV engineers are stuck on single images. 2030 systems (robots, AVs, AR) need temporal understanding: "what will happen NEXT in this video?"

**What you build:** A transformer-based video model that watches partial video clips and predicts the next action with uncertainty quantification.

---

### _ PROJECT F -- Synthetic Data Engine for CV Training
**Difficulty:** ____ | **Impact:** Industry disruption  
**Why it's impressive:** Real-world data collection is expensive, slow, and biased. Synthetic data generation (using rendering engines + domain randomization) is how Tesla, NVIDIA, and Waymo train at scale.

**What you build:** A procedural rendering pipeline using Blender Python API that generates thousands of labeled training images (with ground truth bounding boxes) for any object of interest.

---

### _ PROJECT G -- Vision-Language-Action (VLA) Robot Perception
**Difficulty:** _____ | **Impact:** Robotics frontier  
**Why it's impressive:** RT-2, OpenVLA, and pi0 are the biggest breakthroughs in robotics. Understanding how language guides physical robot actions via vision is the next 10 years.

**What you build:** A simulation environment where a robot arm, guided by natural language commands, uses visual perception to locate and "grasp" objects.

---

### _ PROJECT H -- Neuromorphic Edge CV on Simulated Spiking Networks
**Difficulty:** _____ | **Impact:** Next-gen hardware  
**Why it's impressive:** Traditional CNNs are power-hungry. Intel Loihi, IBM NorthPole, and BrainScaleS are neuromorphic chips that process vision at 1/1000th the energy. Almost nobody knows this exists.

**What you build:** A spiking neural network (SNN) using SpikingJelly that processes image streams as spike trains, demonstrating energy efficiency vs standard CNN.

---

In [ ]:
# ============================================================
# PROJECT A -- Multimodal Visual Question Answering (VQA) Engine
# ============================================================
#
# Architecture: CLIP image encoder + text encoder => attention fusion
#               => answer generation with spatial grounding
#
# In production: use BLIP-2, LLaVA, or InstructBLIP
# Here: implement the core architecture from scratch to understand it

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image

# ---- CORE VQA ARCHITECTURE ----

class CrossModalAttention(nn.Module):
    '''
    Cross-modal attention: allows question (text) to attend to image regions.
    
    This is the key mechanism in VQA:
    - Question = Query (what are we looking for?)
    - Image patches = Keys + Values (where is it?)
    - Output = image regions most relevant to the question
    '''
    def __init__(self, embed_dim=256, num_heads=8):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)
        
    def forward(self, question_feat, image_patches):
        '''
        question_feat:  (batch, 1, embed_dim)      = the question embedding
        image_patches:  (batch, n_patches, embed_dim) = spatial image features
        
        Returns:
            attended:       fused representation
            attention_weights: which patches the question attended to
        '''
        # Question attends to image patches
        # Q = question, K = V = image patches
        attended, attn_weights = self.attention(
            query=question_feat,
            key=image_patches,
            value=image_patches
        )
        return self.norm(attended), attn_weights


class MultimodalVQAModel(nn.Module):
    '''
    Lightweight VQA model that demonstrates the core concept.
    
    Pipeline:
    1. Encode image => spatial feature grid
    2. Encode question => text embedding  
    3. Cross-modal attention (question queries image regions)
    4. Fuse => predict answer category
    5. Return attention map for spatial grounding (Grad-CAM style)
    '''
    
    def __init__(self, vocab_size=100, embed_dim=256, n_answers=20, grid_size=7):
        super().__init__()
        self.grid_size = grid_size
        self.n_patches = grid_size * grid_size
        self.embed_dim = embed_dim
        
        # ---- IMAGE ENCODER (simplified CNN backbone) ----
        self.image_encoder = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64),
            nn.MaxPool2d(2),   # 224 => 112
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
            nn.MaxPool2d(2),   # 112 => 56
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(256),
            nn.MaxPool2d(2),   # 56 => 28
            nn.Conv2d(256, embed_dim, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((grid_size, grid_size))  # => (7, 7)
        )
        
        # Spatial position embedding (tells model WHERE each patch is)
        self.pos_embedding = nn.Parameter(torch.randn(1, self.n_patches, embed_dim))
        
        # ---- QUESTION ENCODER (simple word embedding + GRU) ----
        self.word_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.question_gru = nn.GRU(embed_dim, embed_dim, batch_first=True, bidirectional=False)
        
        # ---- CROSS-MODAL ATTENTION ----
        self.cross_attention = CrossModalAttention(embed_dim, num_heads=8)
        
        # ---- ANSWER PREDICTION HEAD ----
        self.answer_head = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, n_answers)
        )
        
    def forward(self, image, question_tokens):
        '''
        image:           (batch, 3, H, W)
        question_tokens: (batch, seq_len) -- tokenized question
        
        Returns: logits (batch, n_answers), attn_weights (batch, n_patches)
        '''
        B = image.shape[0]
        
        # 1. Encode image => spatial patches
        img_feat = self.image_encoder(image)          # (B, embed_dim, 7, 7)
        img_feat = img_feat.flatten(2).transpose(1,2) # (B, 49, embed_dim)
        img_feat = img_feat + self.pos_embedding       # add positional info
        
        # 2. Encode question => fixed-size embedding
        q_embed = self.word_embed(question_tokens)         # (B, seq, embed_dim)
        _, q_hidden = self.question_gru(q_embed)           # q_hidden: (1, B, embed_dim)
        q_feat = q_hidden.squeeze(0).unsqueeze(1)          # (B, 1, embed_dim)
        
        # 3. Cross-modal attention: question finds relevant image regions
        fused, attn_weights = self.cross_attention(q_feat, img_feat)
        fused = fused.squeeze(1)                           # (B, embed_dim)
        
        # 4. Global image feature (for context)
        global_img = img_feat.mean(dim=1)                 # (B, embed_dim)
        
        # 5. Concatenate and predict answer
        combined = torch.cat([fused, global_img], dim=-1)  # (B, embed_dim*2)
        logits = self.answer_head(combined)                 # (B, n_answers)
        
        return logits, attn_weights.squeeze(1)             # return attention for visualization

# ---- DEMO: VQA INFERENCE WITH SPATIAL GROUNDING ----

def demo_vqa_system():
    '''Demonstrate the complete VQA pipeline with visualization.'''
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Create model
    model = MultimodalVQAModel(vocab_size=50, embed_dim=256, n_answers=10, grid_size=7)
    model.eval()
    
    # Vocabulary (simplified)
    VOCAB = {'<pad>':0, 'what':1, 'is':2, 'color':3, 'the':4, 'object':5,
             'how':6, 'many':7, 'objects':8, 'are':9, 'there':10, 'where':11,
             'red':12, 'blue':13, 'green':14, 'large':15, 'small':16, 'in':17}
    
    ANSWERS = {0:'red', 1:'blue', 2:'green', 3:'one', 4:'two', 5:'three',
               6:'left', 7:'right', 8:'center', 9:'unknown'}
    
    def tokenize(question, max_len=8):
        tokens = [VOCAB.get(w.lower().strip('?'), 0) for w in question.split()]
        tokens = tokens[:max_len] + [0] * (max_len - len(tokens))
        return torch.tensor(tokens).unsqueeze(0)
    
    # Create synthetic test images
    test_cases = []
    
    # Scene 1: Red object on left
    img1 = np.ones((224, 224, 3), dtype=np.uint8) * 200
    cv2.circle(img1, (56, 112), 40, (0, 0, 255), -1)   # red circle, left side (BGR)
    cv2.rectangle(img1, (160, 80), (200, 150), (0, 200, 0), -1)  # green right
    test_cases.append((img1, "What color is the large object?", 0))  # answer: red
    
    # Scene 2: Blue object in center
    img2 = np.ones((224, 224, 3), dtype=np.uint8) * 220
    cv2.rectangle(img2, (80, 70), (150, 160), (255, 0, 0), -1)  # blue center (BGR)
    cv2.circle(img2, (20, 20), 15, (0, 255, 0), -1)  # green corner
    test_cases.append((img2, "What color is the large object?", 1))  # answer: blue
    
    # Scene 3: Multiple objects
    img3 = np.ones((224, 224, 3), dtype=np.uint8) * 210
    for x, c in [(40, (0,0,200)), (112, (0,200,0)), (184, (200,0,0))]:
        cv2.circle(img3, (x, 112), 25, c, -1)
    test_cases.append((img3, "How many objects are there?", 5))  # answer: three
    
    # Run inference and visualize
    fig, axes = plt.subplots(3, 4, figsize=(18, 14))
    fig.suptitle("PROJECT A: Multimodal VQA Engine
Image => Question => Grounded Answer",
                 fontsize=14, fontweight='bold', y=0.98)
    
    for row, (img_np, question, true_idx) in enumerate(test_cases):
        # Prepare tensors
        img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).float() / 255.0
        img_tensor = img_tensor.unsqueeze(0)
        q_tokens = tokenize(question)
        
        # Inference
        with torch.no_grad():
            logits, attn_map = model(img_tensor, q_tokens)
        
        probs = F.softmax(logits, dim=-1)[0]
        pred_idx = probs.argmax().item()
        
        # Reshape attention to spatial grid
        attn_spatial = attn_map[0].reshape(7, 7).numpy()
        attn_spatial = (attn_spatial - attn_spatial.min()) / (attn_spatial.max() - attn_spatial.min() + 1e-8)
        
        # Upsample attention to image size
        attn_upsampled = cv2.resize(attn_spatial, (224, 224), interpolation=cv2.INTER_CUBIC)
        attn_upsampled = np.clip(attn_upsampled, 0, 1)
        
        # Create heatmap overlay
        heatmap = cv2.applyColorMap(np.uint8(255 * attn_upsampled), cv2.COLORMAP_JET)
        heatmap_rgb = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        overlay = (img_np * 0.5 + heatmap_rgb * 0.5).astype(np.uint8)
        
        # Plot
        axes[row, 0].imshow(img_np)
        axes[row, 0].set_title(f"Input Image", fontweight='bold', fontsize=9)
        axes[row, 0].axis('off')
        
        axes[row, 1].imshow(attn_spatial, cmap='hot')
        axes[row, 1].set_title(f"Attention Map
(7x7 grid)", fontweight='bold', fontsize=9)
        axes[row, 1].axis('off')
        
        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title(f"Grounded Attention
(where model looked)", fontweight='bold', fontsize=9)
        axes[row, 2].axis('off')
        
        # Answer probability bar chart
        top5_probs = probs[:6].numpy()
        top5_labels = [ANSWERS[i] for i in range(6)]
        colors = ['#e74c3c' if i == pred_idx else '#3498db' for i in range(6)]
        axes[row, 3].barh(top5_labels, top5_probs, color=colors)
        axes[row, 3].set_title(f"Q: {question}
Pred: '{ANSWERS[pred_idx]}'",
                               fontweight='bold', fontsize=8)
        axes[row, 3].set_xlim(0, 1)
        
    plt.tight_layout()
    plt.show()
    
    print("\n_ VQA Architecture Summary:")
    print("  Image => CNN => 7x7 spatial patches (49 visual tokens)")
    print("  Question => GRU => single question vector")  
    print("  Cross-attention: question queries each image patch")
    print("  Attention weights => spatial heatmap = GROUNDING")
    print("  Fused features => predict answer from vocabulary")
    print()
    print("  Production upgrades:")
    print("  - Replace CNN with CLIP ViT-L/14 (336px)")
    print("  - Replace GRU with BERT/LLaMA text encoder")
    print("  - Replace classification head with LLM decoder (generative VQA)")
    print("  - Scale: BLIP-2 (12B), LLaVA-1.6 (34B), GPT-4V")

demo_vqa_system()


In [ ]:
# ============================================================
# PROJECT B -- Self-Supervised Visual Representation Learning (DINO-style)
# ============================================================
#
# DINO (Self-DIstillation with NO labels) - Facebook AI Research, 2021
# Key insight: A student network and a teacher network look at different
#              augmented views of the same image. Teacher is a slowly
#              updated copy of student (exponential moving average).
#              Self-supervised signal: student output should match teacher output.
#
# This is how Meta trains DINOv2 -- without a single human label!

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import random

# ---- MULTICROP AUGMENTATION ----
# The secret weapon of self-supervised learning:
# Create multiple "views" of the same image at different scales
# Model must learn invariant features to predict across views

class MultiCropAugmentation:
    '''
    DINO-style multi-crop strategy:
    - 2 GLOBAL crops (large portion of image, 224x224)
    - N LOCAL crops  (small portions, 96x96)
    
    Teacher sees: global crops only
    Student sees: all crops
    
    This forces the student to learn from local-global consistency:
    "A small patch of a cat should look like the same cat globally"
    '''
    
    def __init__(self, n_local_crops=4):
        # Strong augmentations for self-supervised (much stronger than supervised)
        self.global_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomResizedCrop(96, scale=(0.6, 1.0)),  # large crop
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.4, 0.4, 0.2, 0.1),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        
        self.local_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomResizedCrop(48, scale=(0.1, 0.4)),  # small crop
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.4, 0.4, 0.2, 0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        
        self.n_local_crops = n_local_crops
    
    def __call__(self, image_np):
        crops = []
        # 2 global crops (for teacher)
        crops.append(self.global_transform(image_np))
        crops.append(self.global_transform(image_np))
        # N local crops (student only)
        for _ in range(self.n_local_crops):
            crops.append(self.local_transform(image_np))
        return crops


# ---- DINO LOSS ----

class DINOLoss(nn.Module):
    '''
    DINO Self-Distillation Loss.
    
    student_output: features from student (all crops)
    teacher_output: features from teacher (global crops only)
    
    Loss = cross-entropy(softmax(teacher), softmax(student))
    
    Key trick: teacher uses centering (subtract running mean) to prevent
    collapse (all outputs becoming identical).
    '''
    
    def __init__(self, out_dim=256, teacher_temp=0.04, student_temp=0.1):
        super().__init__()
        self.teacher_temp = teacher_temp
        self.student_temp = student_temp
        # Center = running average of teacher outputs (prevents mode collapse)
        self.register_buffer('center', torch.zeros(1, out_dim))
    
    def forward(self, student_output, teacher_output):
        '''
        student_output: list of tensors for each crop
        teacher_output: list of tensors for global crops only
        '''
        # Teacher: center + sharpen (low temperature = more confident)
        teacher_out = [(t - self.center) / self.teacher_temp for t in teacher_output]
        teacher_soft = [F.softmax(t, dim=-1).detach() for t in teacher_out]
        
        total_loss = 0
        n_loss_terms = 0
        
        # Student: all crops try to predict teacher's view
        for s_crop in student_output:
            s_soft = F.log_softmax(s_crop / self.student_temp, dim=-1)
            for t_soft in teacher_soft:
                # Resize if needed (global vs local crop different sizes)
                if s_soft.shape == t_soft.shape:
                    loss = -(t_soft * s_soft).sum(dim=-1).mean()
                    total_loss += loss
                    n_loss_terms += 1
        
        # Update center (exponential moving average)
        batch_center = torch.cat(teacher_output).mean(0, keepdim=True)
        self.center = self.center * 0.9 + batch_center * 0.1
        
        return total_loss / max(n_loss_terms, 1)


# ---- MINI DINO BACKBONE ----

class TinyViT(nn.Module):
    '''
    Tiny ViT backbone for DINO demo.
    In production: use ViT-S/16, ViT-B/16, or ViT-L/16.
    '''
    def __init__(self, img_size=96, patch_size=8, in_chans=3,
                 embed_dim=192, depth=6, num_heads=6, out_dim=256):
        super().__init__()
        self.patch_size = patch_size
        n_patches = (img_size // patch_size) ** 2
        
        # Patch embedding: image => sequence of patch tokens
        self.patch_embed = nn.Sequential(
            nn.Conv2d(in_chans, embed_dim, patch_size, patch_size),
            nn.Flatten(2),
        )
        
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        
        # Simplified transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*4,
            dropout=0.0, activation='gelu', batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        
        # Projection head (DINO-specific: multi-layer MLP)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, out_dim),
        )
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    
    def forward(self, x):
        B = x.shape[0]
        # Patch embedding
        patches = self.patch_embed(x).transpose(1, 2)  # (B, n_patches, embed_dim)
        # Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, patches], dim=1) + self.pos_embed
        # Transformer
        x = self.encoder(x)
        x = self.norm(x)
        # CLS token output => projection head
        cls_out = x[:, 0]
        return self.head(cls_out)


# ---- EXPONENTIAL MOVING AVERAGE (Teacher update) ----

def update_teacher(student, teacher, momentum=0.996):
    '''
    Teacher weights = slow-moving average of student weights.
    This prevents the teacher from collapsing to trivial solutions.
    
    teacher_params = momentum * teacher_params + (1-momentum) * student_params
    '''
    with torch.no_grad():
        for s_param, t_param in zip(student.parameters(), teacher.parameters()):
            t_param.data = momentum * t_param.data + (1 - momentum) * s_param.data


# ---- FULL DINO TRAINING DEMO ----

def demo_dino_training():
    '''
    Demonstrate one DINO training iteration with synthetic data.
    Shows the complete self-supervised learning pipeline.
    '''
    torch.manual_seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    OUT_DIM = 128
    
    # Create student and teacher (teacher = copy of student, no gradient)
    student = TinyViT(img_size=96, patch_size=8, embed_dim=64, depth=3, num_heads=4, out_dim=OUT_DIM)
    teacher = TinyViT(img_size=96, patch_size=8, embed_dim=64, depth=3, num_heads=4, out_dim=OUT_DIM)
    teacher.load_state_dict(student.state_dict())
    
    # Teacher never receives gradients
    for param in teacher.parameters():
        param.requires_grad = False
    
    student, teacher = student.to(device), teacher.to(device)
    
    dino_loss_fn = DINOLoss(out_dim=OUT_DIM).to(device)
    optimizer = torch.optim.AdamW(student.parameters(), lr=5e-4, weight_decay=0.04)
    augment = MultiCropAugmentation(n_local_crops=4)
    
    # Simulate unlabeled image dataset (random images for demo)
    print("DINO Self-Supervised Training (No Labels!)")
    print("=" * 55)
    
    losses = []
    n_steps = 30
    
    for step in range(n_steps):
        # Simulate a batch of unlabeled images
        batch_images = [np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8) for _ in range(4)]
        
        # Apply multi-crop augmentation to each image
        all_crops = [[] for _ in range(6)]  # 2 global + 4 local
        for img in batch_images:
            crops = augment(img)
            for i, crop in enumerate(crops):
                all_crops[i].append(crop)
        
        # Stack into batches
        all_crops = [torch.stack(c).to(device) for c in all_crops]
        global_crops = all_crops[:2]  # teacher sees only global
        all_crops_list = all_crops    # student sees all
        
        # Resize local crops to global size for the tiny model
        global_size_crops = []
        for crop in all_crops_list:
            if crop.shape[-1] != 96:
                crop = F.interpolate(crop, size=96, mode='bilinear', align_corners=False)
            global_size_crops.append(crop)
        
        # Forward passes
        # Student processes ALL crops
        student_outs = [student(crop) for crop in global_size_crops]
        
        # Teacher processes only GLOBAL crops (no gradient!)
        with torch.no_grad():
            teacher_outs = [teacher(crop) for crop in [global_size_crops[0], global_size_crops[1]]]
        
        # Compute DINO loss
        loss = dino_loss_fn(student_outs, teacher_outs)
        
        # Update student
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 3.0)
        optimizer.step()
        
        # Update teacher via EMA (NO gradient, NO optimizer)
        momentum = 0.996 + 0.004 * (step / n_steps)  # momentum increases during training
        update_teacher(student, teacher, momentum)
        
        losses.append(loss.item())
        if step % 5 == 0:
            print(f"  Step {step:3d}/{n_steps} | Loss: {loss.item():.4f} | "
                  f"Teacher momentum: {momentum:.4f}")
    
    # Demonstrate learned feature properties
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Loss curve
    axes[0].plot(losses, 'b-', linewidth=2)
    axes[0].set_title("DINO Training Loss
(no labels used!)", fontweight='bold')
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Self-distillation Loss")
    axes[0].grid(alpha=0.3)
    
    # Show feature similarity between views of same image
    student.eval()
    test_img = np.random.randint(50, 200, (128, 128, 3), dtype=np.uint8)
    crops_same = augment(test_img)
    crops_diff_img = augment(np.random.randint(100, 255, (128, 128, 3), dtype=np.uint8))
    
    def get_feat(crop):
        t = F.interpolate(crop.unsqueeze(0).to(device), size=96, mode='bilinear', align_corners=False)
        with torch.no_grad():
            f = student(t)
        return F.normalize(f, dim=-1)
    
    same_sims = []
    for i in range(len(crops_same)):
        for j in range(i+1, len(crops_same)):
            sim = (get_feat(crops_same[i]) * get_feat(crops_same[j])).sum().item()
            same_sims.append(sim)
    
    diff_sims = []
    for ci in crops_same[:3]:
        for cj in crops_diff_img[:3]:
            sim = (get_feat(ci) * get_feat(cj)).sum().item()
            diff_sims.append(sim)
    
    axes[1].hist(same_sims, bins=15, alpha=0.7, color='green', label='Same image views')
    axes[1].hist(diff_sims, bins=15, alpha=0.7, color='red', label='Different images')
    axes[1].set_title("Feature Similarity Distribution
Same image vs Different image", fontweight='bold')
    axes[1].set_xlabel("Cosine Similarity")
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    # Architecture diagram
    axes[2].axis('off')
    arch_text = '''
DINO Architecture

_______________________________
_    UNLABELED IMAGES         _
_    (No human labels!)       _
_______________________________
              _
       Multi-crop augmentation
       _______________
  Global1  Global2  Local1..4
       _      _        _
  __________________   _
  _   TEACHER ViT  _   _    = No gradient
  _ (frozen, EMA)  _   _
  __________________   _
       Teacher targets  _
              _         _
  __________________________
  _     STUDENT ViT        _ = Trained
  __________________________
              _
    DINO Loss (cross-entropy
    student=>teacher predictions)
              _
    EMA update teacher weights

Result: Rich visual features
        without ANY labels!
    '''
    axes[2].text(0.05, 0.95, arch_text, transform=axes[2].transAxes,
                verticalalignment='top', fontfamily='monospace', fontsize=8.5,
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))
    axes[2].set_title("DINO Architecture", fontweight='bold')
    
    plt.suptitle("PROJECT B: DINO Self-Supervised Learning (Zero Labels Required)",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\n_ Key insight: Same cosine similarity gap shows model learned")
    print("   meaningful structure purely from self-supervised signal!")
    print()
    print("Production DINO results (DINOv2):")
    print("  86.5% ImageNet top-1 with frozen features (no fine-tuning!)")
    print("  State-of-the-art on depth estimation, segmentation, retrieval")
    print("  Foundation model: one model, many tasks, zero labeled data")

demo_dino_training()


In [ ]:
# ============================================================
# PROJECT D -- Privacy-Preserving CV with Federated Learning
# ============================================================
#
# Scenario: 4 hospitals want to train a medical image classifier
# Problem: Patient data is private -- cannot leave the hospital
# Solution: Each hospital trains locally, shares only model GRADIENTS
#
# This is how Google Keyboard predicts next word on your phone
# without your texts ever leaving your device!

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

# ---- GLOBAL MODEL (the shared model that all hospitals contribute to) ----

class MedicalImageCNN(nn.Module):
    '''
    Lightweight CNN for medical image classification.
    In federated learning, all clients share this architecture.
    '''
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*4*4, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))


# ---- HOSPITAL (FEDERATED CLIENT) ----

class Hospital:
    '''
    Represents one federated learning client (hospital).
    
    Each hospital:
    - Has its OWN private dataset (never shared)
    - Receives the global model weights
    - Trains locally for a few epochs
    - Sends back only weight UPDATES (deltas)
    - NEVER shares raw patient images
    '''
    
    def __init__(self, hospital_id, n_samples, class_distribution, device):
        self.id = hospital_id
        self.device = device
        self.n_samples = n_samples
        
        # Each hospital has different patient demographics (data heterogeneity)
        # This is a KEY challenge in federated learning
        self.dataset = self._generate_private_data(n_samples, class_distribution)
        
        print(f"  Hospital {hospital_id}: {n_samples} patients | "
              f"Distribution: {class_distribution}")
    
    def _generate_private_data(self, n, class_dist):
        '''Simulate hospital's private medical imaging dataset.'''
        images, labels = [], []
        classes = len(class_dist)
        
        for c, frac in enumerate(class_dist):
            n_class = int(n * frac)
            for _ in range(n_class):
                # Simulate X-ray-like images with class-specific patterns
                img = np.random.normal(0.3, 0.1, (1, 32, 32)).astype(np.float32)
                # Each class has distinctive intensity regions
                intensity = 0.3 + c * 0.25
                r1, r2 = 5 + c*3, 20 - c*2
                img[0, r1:r2, r1:r2] += np.random.normal(intensity, 0.05, (r2-r1, r2-r1))
                img = np.clip(img, 0, 1)
                images.append(img)
                labels.append(c)
        
        images = torch.tensor(np.array(images))
        labels = torch.tensor(labels)
        dataset = torch.utils.data.TensorDataset(images, labels)
        return torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)
    
    def local_train(self, global_model, n_local_epochs=3, lr=0.01):
        '''
        Train locally on private data.
        Returns weight UPDATES (delta = new_weights - old_weights).
        RAW DATA NEVER LEAVES THE HOSPITAL.
        '''
        # Download global model
        local_model = deepcopy(global_model).to(self.device)
        initial_weights = {k: v.clone() for k, v in local_model.state_dict().items()}
        
        optimizer = optim.SGD(local_model.parameters(), lr=lr, momentum=0.9)
        criterion = nn.CrossEntropyLoss()
        
        local_model.train()
        epoch_losses = []
        
        for epoch in range(n_local_epochs):
            total_loss = 0
            for images, labels in self.dataset:
                images, labels = images.to(self.device), labels.to(self.device)
                optimizer.zero_grad()
                loss = criterion(local_model(images), labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            epoch_losses.append(total_loss / len(self.dataset))
        
        # Compute weight DELTA (what the hospital contributes to global model)
        weight_delta = {}
        for k, v in local_model.state_dict().items():
            weight_delta[k] = v - initial_weights[k]
        
        # Return delta (NOT raw data, NOT full model -- just updates)
        return weight_delta, np.mean(epoch_losses), len(self.dataset.dataset)
    
    def evaluate(self, model):
        '''Evaluate global model on local data.'''
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in self.dataset:
                images, labels = images.to(self.device), labels.to(self.device)
                preds = model(images).argmax(dim=1)
                correct += preds.eq(labels).sum().item()
                total += labels.size(0)
        return 100. * correct / total


# ---- FEDERATED SERVER ----

class FederatedServer:
    '''
    Central coordinator (the 'cloud server').
    
    Responsibilities:
    - Maintain the global model
    - Aggregate updates from hospitals (FedAvg algorithm)
    - Distribute updated model back to hospitals
    
    NEVER sees raw patient data -- only model weight updates.
    '''
    
    def __init__(self, num_classes=3, device='cpu'):
        self.device = device
        self.global_model = MedicalImageCNN(num_classes).to(device)
        self.round_history = []
    
    def aggregate(self, client_updates, client_sizes):
        '''
        FedAvg (Federated Averaging) -- McMahan et al., 2017
        
        Weighted average of client updates:
        global_delta = Sum(n_i / N_total) x delta_i
        
        Clients with more data have proportionally more influence.
        '''
        total_samples = sum(client_sizes)
        
        # Initialize aggregated delta
        aggregated_delta = {k: torch.zeros_like(v)
                           for k, v in self.global_model.state_dict().items()}
        
        for delta, n_samples in zip(client_updates, client_sizes):
            weight = n_samples / total_samples  # hospital's contribution weight
            for k in aggregated_delta:
                aggregated_delta[k] += weight * delta[k]
        
        # Apply aggregated update to global model
        global_state = self.global_model.state_dict()
        for k in global_state:
            global_state[k] = global_state[k] + aggregated_delta[k]
        self.global_model.load_state_dict(global_state)


# ---- RUN FEDERATED LEARNING SIMULATION ----

def run_federated_simulation():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print("FEDERATED LEARNING: Multi-Hospital Medical Imaging")
    print("=" * 58)
    print("PRIVACY GUARANTEE: No hospital shares raw patient data!")
    print("=" * 58)
    
    # Create hospitals with heterogeneous (non-IID) data distributions
    # Real-world: different hospitals see different patient populations
    print("
Initializing hospitals:")
    hospitals = [
        Hospital("A (General)",   n_samples=200, class_distribution=[0.6, 0.3, 0.1], device=device),
        Hospital("B (Cardiology)", n_samples=150, class_distribution=[0.2, 0.6, 0.2], device=device),
        Hospital("C (Oncology)",   n_samples=180, class_distribution=[0.1, 0.2, 0.7], device=device),
        Hospital("D (Rural)",      n_samples=80,  class_distribution=[0.4, 0.4, 0.2], device=device),
    ]
    
    server = FederatedServer(num_classes=3, device=device)
    
    # Also train a centralized baseline (to show federated ~= centralized)
    central_model = MedicalImageCNN(num_classes=3).to(device)
    
    federated_accs = []
    n_rounds = 15
    
    print(f"
Training for {n_rounds} federated rounds...")
    print("-" * 58)
    
    for fed_round in range(n_rounds):
        client_updates = []
        client_sizes = []
        round_losses = []
        
        # Each hospital trains locally
        for hospital in hospitals:
            delta, loss, n = hospital.local_train(server.global_model, n_local_epochs=2)
            client_updates.append(delta)
            client_sizes.append(n)
            round_losses.append(loss)
        
        # Server aggregates updates (FedAvg)
        server.aggregate(client_updates, client_sizes)
        
        # Evaluate global model on each hospital's data
        round_accs = [h.evaluate(server.global_model) for h in hospitals]
        avg_acc = np.mean(round_accs)
        federated_accs.append(avg_acc)
        
        if fed_round % 3 == 0:
            print(f"  Round {fed_round+1:2d}/{n_rounds} | "
                  f"Avg Accuracy: {avg_acc:.1f}% | "
                  f"Per-hospital: {[f'{a:.0f}%' for a in round_accs]}")
    
    # ---- VISUALIZATION ----
    fig, axes = plt.subplots(1, 3, figsize=(16, 6))
    
    # Accuracy curve
    axes[0].plot(federated_accs, 'b-o', linewidth=2, markersize=5)
    axes[0].axhline(federated_accs[-1], color='green', linestyle='--', alpha=0.5)
    axes[0].set_title("Federated Learning Accuracy
Across Training Rounds", fontweight='bold')
    axes[0].set_xlabel("Communication Round")
    axes[0].set_ylabel("Avg Accuracy (%) across hospitals")
    axes[0].grid(alpha=0.3)
    axes[0].set_ylim(0, 105)
    
    # Per-hospital performance
    final_accs = [h.evaluate(server.global_model) for h in hospitals]
    h_names = [h.id for h in hospitals]
    colors = ['#e74c3c','#3498db','#2ecc71','#f39c12']
    bars = axes[1].bar(range(len(hospitals)), final_accs, color=colors, alpha=0.85)
    axes[1].set_xticks(range(len(hospitals)))
    axes[1].set_xticklabels([n.split('(')[1].rstrip(')') for n in h_names], fontsize=9)
    axes[1].set_title("Final Accuracy Per Hospital
(Same global model, different local data)", fontweight='bold')
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].set_ylim(0, 105)
    for bar, acc in zip(bars, final_accs):
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                    f'{acc:.1f}%', ha='center', fontweight='bold', fontsize=10)
    
    # System diagram
    axes[2].axis('off')
    fl_diagram = '''
FEDERATED LEARNING FLOW

Round N:
  SERVER sends global model weights v
  
  ________________________________
  _  Hospital A  _  Hospital B   _
  _  _ Private  _  _ Private   _
  _  patients    _  patients     _
  _  Local train _  Local train  _
  _  deltaweights ^  _  deltaweights ^   _
  ________________________________
         ^ Weight deltas only
         ^ NO raw images shared
  
  SERVER runs FedAvg:
  global_w += Sum(n_/N) x deltaw_
  
  v Updated global model

Privacy properties:
  _ Raw images stay local
  _ Compliant with GDPR
  _ HIPAA compatible
  _ Differential privacy
     can be added to deltas
  _ Secure aggregation
     possible (cryptographic)
    '''
    axes[2].text(0.02, 0.97, fl_diagram, transform=axes[2].transAxes,
                verticalalignment='top', fontfamily='monospace', fontsize=8.5,
                bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.4))
    axes[2].set_title("Federated Learning System", fontweight='bold')
    
    plt.suptitle("PROJECT D: Privacy-Preserving Medical CV -- Federated Learning",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"
_ Final federated accuracy: {federated_accs[-1]:.1f}%")
    print("   Trained on combined knowledge of 4 hospitals")
    print("   WITHOUT any hospital sharing patient data!")

run_federated_simulation()


In [ ]:
# ============================================================
# PROJECT E -- Video Temporal Reasoning & Action Anticipation
# ============================================================
#
# Problem: Given the first 50% of a video, predict what action will
#          happen in the NEXT 50% -- with uncertainty estimates.
#
# This is crucial for: robotics (anticipate human intent),
#                      autonomous vehicles (predict pedestrian trajectory),
#                      smart assistants (anticipate user needs)
#
# Architecture: Visual Frame Encoder + Temporal Transformer + UQ head

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import FancyBboxPatch

# ---- TEMPORAL TRANSFORMER FOR VIDEO UNDERSTANDING ----

class TemporalPositionalEncoding(nn.Module):
    '''
    Learnable temporal position encoding -- tells the model WHEN in the video
    each frame occurs. This is crucial: frame 5 of 10 carries different
    information than frame 5 of 50.
    '''
    def __init__(self, d_model, max_frames=64):
        super().__init__()
        self.pos_embed = nn.Embedding(max_frames, d_model)
        self.time_proj = nn.Linear(1, d_model)  # for continuous time
        
    def forward(self, x, frame_indices=None):
        '''x: (batch, n_frames, d_model)'''
        B, T, D = x.shape
        if frame_indices is None:
            frame_indices = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        pos = self.pos_embed(frame_indices)
        return x + pos


class FrameEncoder(nn.Module):
    '''
    Encodes each individual video frame into a feature vector.
    In production: use VideoMAE or TimeSformer patch encoder.
    '''
    def __init__(self, in_channels=3, frame_size=32, embed_dim=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, stride=2, padding=1), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.GELU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.GELU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        self.proj = nn.Linear(128, embed_dim)
    
    def forward(self, frames):
        '''frames: (batch*n_frames, C, H, W) => (batch*n_frames, embed_dim)'''
        return self.proj(self.conv(frames))


class VideoTransformer(nn.Module):
    '''
    Temporal Transformer for video understanding.
    
    Key design decisions:
    1. Causal masking: only look at PAST frames (autoregressive)
    2. Learnable [FUTURE] token: aggregates temporal context for prediction
    3. Uncertainty head: outputs both mean prediction AND confidence
    '''
    
    def __init__(self, n_actions=8, embed_dim=128, n_frames=16, n_heads=4, n_layers=4):
        super().__init__()
        self.embed_dim = embed_dim
        self.n_actions = n_actions
        
        # Frame feature extractor
        self.frame_encoder = FrameEncoder(embed_dim=embed_dim)
        
        # Temporal position encoding
        self.temporal_pe = TemporalPositionalEncoding(embed_dim, max_frames=n_frames+1)
        
        # [FUTURE] prediction token (analogous to [CLS] in BERT)
        self.future_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        
        # Causal temporal transformer
        # Causal = each position can only attend to earlier positions
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim*4, dropout=0.1,
            batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(embed_dim)
        
        # Action classification head
        self.action_head = nn.Linear(embed_dim, n_actions)
        
        # UNCERTAINTY QUANTIFICATION head
        # Outputs log-variance: model can say "I'm unsure about this prediction"
        # This is crucial for safety-critical systems (autonomous vehicles, robots)
        self.uncertainty_head = nn.Sequential(
            nn.Linear(embed_dim, 64), nn.ReLU(),
            nn.Linear(64, n_actions)  # log-variance per action
        )
    
    def forward(self, frames_batch):
        '''
        frames_batch: (batch, n_frames, C, H, W)
        Returns: action_logits, uncertainty (log-variance), attention_weights
        '''
        B, T, C, H, W = frames_batch.shape
        
        # Encode each frame independently
        frames_flat = frames_batch.view(B*T, C, H, W)
        frame_features = self.frame_encoder(frames_flat)  # (B*T, embed_dim)
        frame_features = frame_features.view(B, T, self.embed_dim)  # (B, T, embed_dim)
        
        # Prepend [FUTURE] token
        future_token = self.future_token.expand(B, -1, -1)  # (B, 1, embed_dim)
        sequence = torch.cat([future_token, frame_features], dim=1)  # (B, T+1, embed_dim)
        
        # Add temporal position encoding
        sequence = self.temporal_pe(sequence)
        
        # Create causal mask (each frame only sees previous frames)
        seq_len = sequence.shape[1]
        causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        causal_mask = causal_mask.to(frames_batch.device)
        
        # Temporal self-attention
        out = self.transformer(sequence, mask=causal_mask)
        out = self.norm(out)
        
        # [FUTURE] token output contains temporal context for prediction
        future_repr = out[:, 0]  # (B, embed_dim)
        
        action_logits = self.action_head(future_repr)
        log_variance = self.uncertainty_head(future_repr)  # uncertainty!
        
        return action_logits, log_variance


# ---- SYNTHETIC VIDEO DATASET ----

class SyntheticVideoDataset:
    '''
    Generates synthetic action videos.
    Each action has a distinct temporal pattern across frames.
    '''
    ACTIONS = ['walking', 'running', 'jumping', 'waving', 
               'sitting', 'standing', 'throwing', 'catching']
    
    def generate_video(self, action_idx, n_frames=16, frame_size=32):
        '''Generate a synthetic video with action-specific motion pattern.'''
        frames = []
        for t in range(n_frames):
            frame = np.zeros((3, frame_size, frame_size), dtype=np.float32)
            
            # Action-specific visual patterns over time
            progress = t / n_frames  # 0 to 1
            
            if action_idx == 0:  # walking: steady horizontal movement
                x = int(2 + 28 * progress)
                y = frame_size // 2
            elif action_idx == 1:  # running: fast horizontal + vertical oscillation
                x = int(2 + 28 * progress)
                y = frame_size//2 + int(4 * np.sin(progress * 8 * np.pi))
            elif action_idx == 2:  # jumping: parabolic trajectory
                x = frame_size // 2
                y = int(28 - 24 * np.sin(progress * np.pi))
            elif action_idx == 3:  # waving: oscillating position
                x = frame_size // 2 + int(8 * np.sin(progress * 6 * np.pi))
                y = frame_size // 3
            elif action_idx == 4:  # sitting: downward movement, stops
                x = frame_size // 2
                y = int(4 + min(progress * 3, 1.0) * 20)
            elif action_idx == 5:  # standing: upward movement
                x = frame_size // 2
                y = int(28 - min(progress * 3, 1.0) * 20)
            elif action_idx == 6:  # throwing: arm extension
                x = int(4 + 20 * progress)
                y = int(frame_size//2 - 8 * progress)
            else:  # catching: reaching motion
                x = int(frame_size//2 + 12 * np.cos(progress * np.pi * 0.5))
                y = int(frame_size//2 + 12 * np.sin(progress * np.pi * 0.5))
            
            x, y = np.clip(x, 2, frame_size-3), np.clip(y, 2, frame_size-3)
            
            # Draw "person" (colored blob) at computed position
            color_base = np.array([0.7+0.3*action_idx/7, 0.3+0.5*progress, 0.5])
            frame[0, y-2:y+3, x-2:x+3] = color_base[0]  # R channel
            frame[1, y-2:y+3, x-2:x+3] = color_base[1]  # G channel
            frame[2, y-2:y+3, x-2:x+3] = color_base[2]  # B channel
            
            # Add background noise
            frame += np.random.normal(0, 0.02, frame.shape).astype(np.float32)
            frame = np.clip(frame, 0, 1)
            frames.append(frame)
        
        return np.stack(frames)  # (n_frames, 3, H, W)
    
    def get_batch(self, batch_size=8, n_frames=16, frame_size=32):
        '''Return a batch of (video, label) pairs.'''
        videos, labels = [], []
        for _ in range(batch_size):
            action = np.random.randint(0, len(self.ACTIONS))
            video = self.generate_video(action, n_frames, frame_size)
            videos.append(video)
            labels.append(action)
        return (torch.tensor(np.stack(videos), dtype=torch.float32),
                torch.tensor(labels))


# ---- TRAIN AND EVALUATE ----

def demo_temporal_reasoning():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dataset = SyntheticVideoDataset()
    
    N_FRAMES = 16
    N_ACTIONS = 8
    OBSERVED_FRAMES = N_FRAMES // 2  # model sees first HALF of video
    
    model = VideoTransformer(n_actions=N_ACTIONS, embed_dim=64, 
                             n_frames=N_FRAMES, n_heads=4, n_layers=3).to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    
    print("Video Temporal Reasoning & Action Anticipation")
    print("=" * 54)
    print(f"Model sees: first {OBSERVED_FRAMES}/{N_FRAMES} frames ({OBSERVED_FRAMES/N_FRAMES:.0%} of video)")
    print(f"Task: predict action label + confidence for remaining frames")
    print()
    
    losses, accs = [], []
    N_STEPS = 40
    
    for step in range(N_STEPS):
        model.train()
        videos, labels = dataset.get_batch(batch_size=16, n_frames=N_FRAMES, frame_size=32)
        
        # Only give model the first half of each video (anticipation task!)
        partial_videos = videos[:, :OBSERVED_FRAMES].to(device)
        labels = labels.to(device)
        
        logits, log_var = model(partial_videos)
        
        # Negative log-likelihood loss with uncertainty
        # This rewards the model for being confident AND accurate
        variance = torch.exp(log_var)
        nll_loss = ((logits - F.one_hot(labels, N_ACTIONS).float()) ** 2 / (2 * variance + 1e-6) 
                    + 0.5 * log_var).mean()
        ce_loss = F.cross_entropy(logits, labels)
        loss = ce_loss + 0.1 * nll_loss  # combined loss
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        preds = logits.argmax(dim=1)
        acc = preds.eq(labels).float().mean().item()
        losses.append(loss.item())
        accs.append(acc * 100)
        
        if step % 8 == 0:
            print(f"  Step {step:3d}/{N_STEPS} | Loss: {loss.item():.4f} | Acc: {acc:.1%}")
    
    # ---- VISUALIZATION ----
    model.eval()
    test_videos, test_labels = dataset.get_batch(batch_size=4, n_frames=N_FRAMES, frame_size=32)
    
    fig = plt.figure(figsize=(18, 14))
    fig.suptitle("PROJECT E: Video Temporal Reasoning & Action Anticipation
"
                 f"Model observes first {OBSERVED_FRAMES}/{N_FRAMES} frames => predicts action + uncertainty",
                 fontsize=13, fontweight='bold')
    
    # Training curves
    ax1 = fig.add_subplot(3, 4, 1)
    ax1.plot(losses, 'b-', linewidth=1.5, alpha=0.8, label='Loss')
    ax1.set_title("Training Loss", fontweight='bold', fontsize=9)
    ax1.set_xlabel("Step"); ax1.grid(alpha=0.3)
    
    ax2 = fig.add_subplot(3, 4, 2)
    ax2.plot(accs, 'g-', linewidth=1.5)
    ax2.set_title("Training Accuracy", fontweight='bold', fontsize=9)
    ax2.set_xlabel("Step"); ax2.set_ylim(0, 105); ax2.grid(alpha=0.3)
    
    # Test cases with uncertainty
    with torch.no_grad():
        partial_test = test_videos[:4, :OBSERVED_FRAMES].to(device)
        logits, log_vars = model(partial_test)
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        uncertainties = torch.exp(log_vars).cpu().numpy()
    
    for i in range(4):
        # Show observed frames (first 3 only for display)
        for f_idx in range(3):
            ax = fig.add_subplot(3, 4, 5 + (i * 1 if i < 2 else i))
            frame_np = test_videos[i, f_idx].permute(1, 2, 0).numpy()
            ax.imshow(frame_np)
            ax.set_title(f"Sample {i+1}: Frame {f_idx+1}", fontsize=7)
            ax.axis('off')
            break  # just show first frame per sample for space
        
        # Prediction + uncertainty bars
        ax = fig.add_subplot(3, 4, 9 + i)
        pred_probs = probs[i]
        pred_unc = uncertainties[i].mean(-1) if uncertainties[i].ndim > 1 else uncertainties[i]
        true_label = test_labels[i].item()
        pred_label = pred_probs.argmax()
        
        colors = ['green' if j == true_label else ('red' if j == pred_label else 'steelblue')
                  for j in range(N_ACTIONS)]
        bars = ax.bar(range(N_ACTIONS), pred_probs, color=colors, alpha=0.75)
        ax.set_xticks(range(N_ACTIONS))
        ax.set_xticklabels([a[:4] for a in dataset.ACTIONS], rotation=45, fontsize=7)
        ax.set_ylim(0, 1)
        
        correct = "_" if pred_label == true_label else "_"
        avg_uncertainty = uncertainties[i].mean()
        ax.set_title(f"Sample {i+1}: {correct} True:{dataset.ACTIONS[true_label][:5]}
"
                    f"Pred:{dataset.ACTIONS[pred_label][:5]} | Unc:{avg_uncertainty:.3f}",
                    fontsize=7, fontweight='bold')
        ax.grid(alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print(f"
_ Final accuracy: {np.mean(accs[-5:]):.1f}% on action anticipation")
    print()
    print("Why uncertainty matters (safety-critical systems):")
    print("  Low uncertainty + correct prediction  => Act confidently")
    print("  High uncertainty + any prediction     => Ask for human help")
    print("  Low uncertainty + wrong prediction    => MODEL FAILURE (dangerous!)")
    print()
    print("Extensions for production:")
    print("  - Replace FrameEncoder with VideoMAE pre-trained ViT")
    print("  - Use 3D spatiotemporal convolutions (SlowFast, X3D)")
    print("  - Add Monte Carlo Dropout for better UQ")
    print("  - Train on Kinetics-700, Epic-Kitchens, Ego4D datasets")

demo_temporal_reasoning()


In [ ]:
# ============================================================
# PROJECT F -- Synthetic Data Engine for CV Training
# ============================================================
#
# Tesla, NVIDIA DRIVE, Waymo -- all use synthetic data at massive scale.
# "Domain randomization": vary lighting, textures, viewpoints so the
# model becomes robust to real-world variation.
#
# This demo: procedural generation of labeled training data with
# domain randomization -- zero manual annotation required!

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from dataclasses import dataclass
from typing import List, Tuple
import json

@dataclass
class BoundingBox:
    '''Ground truth annotation -- automatically generated from renderer.'''
    class_name: str
    class_id: int
    x1: int; y1: int; x2: int; y2: int
    
    @property
    def area(self): return (self.x2-self.x1) * (self.y2-self.y1)
    
    def to_yolo(self, img_w, img_h):
        '''Convert to YOLO format: class cx cy w h (normalized).'''
        cx = ((self.x1 + self.x2) / 2) / img_w
        cy = ((self.y1 + self.y2) / 2) / img_h
        w  = (self.x2 - self.x1) / img_w
        h  = (self.y2 - self.y1) / img_h
        return f"{self.class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


class DomainRandomizer:
    '''
    Domain randomization engine: generates varied visual conditions
    so models trained on synthetic data transfer to real-world.
    
    Key idea from OpenAI/NVIDIA research: if you randomize enough
    parameters (lighting, texture, scale, rotation, background),
    the real world is 'just another variation' the model handles.
    '''
    
    @staticmethod
    def random_background(h, w):
        '''Generate randomized background (replaces fixed texture).'''
        bg_type = np.random.choice(['gradient', 'noise', 'solid', 'checkerboard'])
        
        if bg_type == 'gradient':
            r = np.linspace(np.random.randint(0,100), np.random.randint(150,255), w)
            g = np.linspace(np.random.randint(0,100), np.random.randint(150,255), w)
            b = np.linspace(np.random.randint(0,100), np.random.randint(150,255), w)
            bg = np.zeros((h, w, 3), dtype=np.uint8)
            bg[:, :, 0] = r[np.newaxis, :]
            bg[:, :, 1] = g[np.newaxis, :]
            bg[:, :, 2] = b[np.newaxis, :]
        elif bg_type == 'noise':
            bg = np.random.randint(50, 200, (h, w, 3), dtype=np.uint8)
        elif bg_type == 'checkerboard':
            bg = np.zeros((h, w, 3), dtype=np.uint8)
            cell = np.random.randint(20, 60)
            c1 = np.random.randint(0, 200, 3)
            c2 = np.random.randint(0, 200, 3)
            for i in range(0, h, cell):
                for j in range(0, w, cell):
                    color = c1 if ((i//cell + j//cell) % 2 == 0) else c2
                    bg[i:i+cell, j:j+cell] = color
        else:
            bg = np.full((h, w, 3), np.random.randint(50, 200, 3), dtype=np.uint8)
        
        return bg
    
    @staticmethod
    def random_lighting(image):
        '''Simulate different lighting conditions.'''
        effect = np.random.choice(['bright', 'dark', 'shadow', 'normal', 'high_contrast'])
        
        img = image.astype(np.float32)
        if effect == 'bright':
            img = np.clip(img * np.random.uniform(1.2, 1.8), 0, 255)
        elif effect == 'dark':
            img = np.clip(img * np.random.uniform(0.3, 0.7), 0, 255)
        elif effect == 'shadow':
            shadow_y = np.random.randint(0, image.shape[0])
            img[shadow_y:] = np.clip(img[shadow_y:] * 0.5, 0, 255)
        elif effect == 'high_contrast':
            img = np.clip((img - 128) * np.random.uniform(1.5, 2.5) + 128, 0, 255)
        
        return img.astype(np.uint8)
    
    @staticmethod
    def add_noise(image):
        '''Add realistic sensor noise (Gaussian + occasional salt-pepper).'''
        img = image.astype(np.float32)
        sigma = np.random.uniform(0, 15)
        img += np.random.normal(0, sigma, img.shape)
        
        if np.random.random() < 0.3:  # sometimes add salt-pepper
            sp_mask = np.random.random(img.shape[:2])
            img[sp_mask < 0.01] = 0
            img[sp_mask > 0.99] = 255
        
        return np.clip(img, 0, 255).astype(np.uint8)


class ProceduralSceneGenerator:
    '''
    Procedural 2D scene generator with automatic bounding box annotation.
    Generates (image, annotations) pairs with zero human labeling.
    
    In production: use Blender, Unreal Engine, CARLA, or NVIDIA Isaac Sim
    for 3D photorealistic rendering with automatic annotation.
    '''
    
    CLASSES = {
        'vehicle': 0,
        'pedestrian': 1, 
        'traffic_sign': 2,
        'obstacle': 3
    }
    
    def __init__(self, img_size=(400, 600)):
        self.H, self.W = img_size
    
    def render_vehicle(self, canvas, x, y, scale=1.0, color_var=True):
        '''Render a stylized vehicle with randomized appearance.'''
        w = int(80 * scale); h = int(40 * scale)
        
        # Random color variation (domain randomization!)
        if color_var:
            base_color = tuple(np.random.randint(50, 200, 3).tolist())
        else:
            base_color = (100, 100, 200)
        
        # Body
        cv2.rectangle(canvas, (x, y), (x+w, y+h), base_color, -1)
        # Windshield
        cv2.rectangle(canvas, (x+w//4, y+5), (x+3*w//4, y+h//2), 
                      (180, 220, 255), -1)
        # Wheels
        wheel_r = max(7, int(10 * scale))
        cv2.circle(canvas, (x+wheel_r+5, y+h), wheel_r, (30,30,30), -1)
        cv2.circle(canvas, (x+w-wheel_r-5, y+h), wheel_r, (30,30,30), -1)
        # Headlights
        cv2.circle(canvas, (x+w-8, y+h//3), 5, (255, 255, 150), -1)
        
        x1, y1 = x, y - wheel_r
        x2, y2 = x + w, y + h + wheel_r
        return BoundingBox('vehicle', 0, x1, y1, x2, y2)
    
    def render_pedestrian(self, canvas, x, y, scale=1.0):
        '''Render a stick-figure pedestrian.'''
        h = int(60 * scale)
        skin = tuple(np.random.randint(120, 200, 3).tolist())
        shirt = tuple(np.random.randint(0, 255, 3).tolist())
        
        # Head
        r = int(8 * scale)
        cv2.circle(canvas, (x, y), r, skin, -1)
        # Body
        cv2.line(canvas, (x, y+r), (x, y+h//2), shirt, int(6*scale))
        # Arms
        cv2.line(canvas, (x, y+int(h*0.25)), (x-int(15*scale), y+int(h*0.4)), 
                 skin, int(4*scale))
        cv2.line(canvas, (x, y+int(h*0.25)), (x+int(15*scale), y+int(h*0.4)), 
                 skin, int(4*scale))
        # Legs
        cv2.line(canvas, (x, y+h//2), (x-int(10*scale), y+h), shirt, int(5*scale))
        cv2.line(canvas, (x, y+h//2), (x+int(10*scale), y+h), shirt, int(5*scale))
        
        margin = int(20 * scale)
        return BoundingBox('pedestrian', 1, x-margin, y-r, x+margin, y+h)
    
    def render_traffic_sign(self, canvas, x, y, scale=1.0):
        '''Render a randomized traffic sign.'''
        sign_type = np.random.choice(['stop', 'yield', 'warning'])
        size = int(30 * scale)
        
        if sign_type == 'stop':
            pts = np.array([[x, y-size], [x+size//2, y-size//2], [x+size//2, y+size//2],
                           [x, y+size], [x-size//2, y+size//2], [x-size//2, y-size//2]], np.int32)
            cv2.fillPoly(canvas, [pts], (0, 0, 220))
            cv2.putText(canvas, 'STOP', (x-size//2+3, y+5), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.3*scale, (255,255,255), 1)
        elif sign_type == 'yield':
            pts = np.array([[x, y+size], [x+size, y-size//2], [x-size, y-size//2]], np.int32)
            cv2.fillPoly(canvas, [pts], (0, 200, 255))
        else:
            pts = np.array([[x, y-size], [x+size, y+size//2], [x-size, y+size//2]], np.int32)
            cv2.fillPoly(canvas, [pts], (0, 220, 220))
        
        return BoundingBox('traffic_sign', 2, x-size, y-size, x+size, y+size)
    
    def generate_scene(self):
        '''Generate one complete synthetic training scene with all annotations.'''
        dr = DomainRandomizer()
        
        # Randomized background (road-like scene)
        canvas = dr.random_background(self.H, self.W)
        
        # Add road (perspective-like)
        road_pts = np.array([[self.W//3, self.H//2], [2*self.W//3, self.H//2],
                              [self.W, self.H], [0, self.H]], np.int32)
        road_color = tuple(np.random.randint(60, 100, 3).tolist())
        cv2.fillPoly(canvas, [road_pts], road_color)
        
        annotations: List[BoundingBox] = []
        
        # Place objects with random positions and scales
        n_vehicles = np.random.randint(1, 4)
        for _ in range(n_vehicles):
            scale = np.random.uniform(0.6, 1.3)
            x = np.random.randint(20, self.W - 120)
            y = np.random.randint(self.H//2, self.H - 60)
            
            # Occlusion check: sometimes objects partially go off-screen (realistic!)
            if x + int(80*scale) > self.W: continue
            
            bbox = self.render_vehicle(canvas, x, y, scale)
            annotations.append(bbox)
        
        n_peds = np.random.randint(0, 3)
        for _ in range(n_peds):
            scale = np.random.uniform(0.5, 1.0)
            x = np.random.randint(30, self.W-30)
            y = np.random.randint(self.H//2+10, self.H-80)
            bbox = self.render_pedestrian(canvas, x, y, scale)
            annotations.append(bbox)
        
        n_signs = np.random.randint(1, 3)
        for _ in range(n_signs):
            scale = np.random.uniform(0.7, 1.2)
            x = np.random.randint(50, self.W-50)
            y = np.random.randint(30, self.H//2)
            bbox = self.render_traffic_sign(canvas, x, y, scale)
            annotations.append(bbox)
        
        # Apply domain randomization effects
        canvas = dr.random_lighting(canvas)
        canvas = dr.add_noise(canvas)
        
        # Random camera blur (simulate focus/motion blur)
        if np.random.random() < 0.3:
            ksize = np.random.choice([3, 5])
            canvas = cv2.GaussianBlur(canvas, (ksize, ksize), 0)
        
        return canvas, annotations
    
    def generate_dataset(self, n_images=12):
        '''Generate a complete annotated dataset.'''
        return [self.generate_scene() for _ in range(n_images)]


# ---- GENERATE AND DISPLAY DATASET ----

gen = ProceduralSceneGenerator(img_size=(300, 450))
dataset = gen.generate_dataset(n_images=6)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
class_colors = {0: (0,255,0), 1: (0,0,255), 2: (255,165,0), 3: (255,0,255)}

total_annotations = 0
print("SYNTHETIC DATA ENGINE -- Dataset Statistics")
print("=" * 50)

for idx, (img, annotations) in enumerate(dataset):
    ax = axes[idx // 3, idx % 3]
    
    # Draw annotations on image
    vis = img.copy()
    for bbox in annotations:
        color = class_colors.get(bbox.class_id, (255,255,255))
        cv2.rectangle(vis, (bbox.x1, bbox.y1), (bbox.x2, bbox.y2), color, 2)
        label = f"{bbox.class_name}"
        cv2.putText(vis, label, (bbox.x1, bbox.y1-5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
    
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Scene {idx+1}: {len(annotations)} objects auto-annotated",
                fontweight='bold', fontsize=9)
    ax.axis('off')
    total_annotations += len(annotations)
    
    # Print YOLO format annotations
    print(f"  Scene {idx+1}: {len(annotations)} objects | "
          f"YOLO labels: {[b.to_yolo(450, 300) for b in annotations[:2]][:1]}...")

legend_elements = [mpatches.Patch(color=np.array(v)/255, label=k) 
                   for k, v in {'Vehicle':(0,255,0),'Pedestrian':(0,0,255),
                                 'Sign':(255,165,0)}.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=10)

plt.suptitle(f"PROJECT F: Synthetic Data Engine -- {len(dataset)} scenes, "
             f"{total_annotations} auto-annotated objects, ZERO human effort",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"
_ Generated {len(dataset)} scenes with {total_annotations} annotations")
print("   Production pipeline: Blender Python API => render 100K scenes overnight")
print("   Domain randomization covers: lighting, scale, color, blur, noise")
print("   Output formats: YOLO, COCO JSON, Pascal VOC XML")
print()
print("Industry scale numbers:")
print("  Tesla:  ~1 billion auto-labeled frames from fleet (Autopilot)")
print("  NVIDIA: 50B synthetic frames for DRIVE AV training")
print("  Meta:   3.5B Instagram images (SSL, no annotation needed)")


---

## 18.3 The 2030 Skill Stack

What separates a **2030-ready CV engineer** from a typical graduate:

```
TIER 1 -- Foundation (table stakes, everyone has this)
__________________________________________________
Python, PyTorch, OpenCV, basic CNN training, transfer learning

TIER 2 -- Differentiated (top 20%)
_________________________________
Vision-Language models (CLIP, BLIP, LLaVA)
Self-supervised learning (DINO, MAE, SimCLR)
Video/temporal understanding
Model optimization (quantization, pruning, distillation)
Uncertainty quantification
Deployment (ONNX, TensorRT, CoreML)

TIER 3 -- Elite (top 1-5%)
___________________________
3D vision (NeRF, Gaussian Splatting, point clouds)
Embodied AI / Vision-Language-Action
Synthetic data pipelines (Blender, Isaac Sim)
Neuromorphic / Spiking Neural Networks
Custom CUDA kernels / hardware-aware ML
Federated learning with differential privacy
Reading and re-implementing ArXiv papers in <48h
```

---

## 18.4 Deployment Pipeline (Production CV)

```
____________________________________________________________________
_                    PRODUCTION CV PIPELINE                        _
_                                                                  _
_  Research ___ Training ___ Optimization ___ Serving ___ Monitor _
_                                                                  _
_  ArXiv        PyTorch      ONNX Export      FastAPI     Drift   _
_  Papers       Lightning    TensorRT          gRPC        detect  _
_  SOTA         HuggingFace  Quantization     Triton      A/B     _
_  reprod.      Weights&B    Pruning          Server      test    _
_               DDP          CoreML           Edge        Alerts  _
_               Mixed Prec.  OpenVINO         Deploy.             _
____________________________________________________________________
```

---

## 18.5 Learning Resources (Curated for 2030)

### _ Books

| Book | Level | Focus |
|------|-------|-------|
| *Deep Learning* -- Goodfellow et al. | Advanced | Theory foundation |
| *Computer Vision: Algorithms and Applications* -- Szeliski | All | Classic CV |
| *Programming Computer Vision with Python* -- Solem | Beginner | Practical |
| *Hands-On Machine Learning* -- G_ron | Intermediate | Applied ML |
| *Probabilistic Machine Learning* -- Murphy | Advanced | Theory + UQ |

### _ Courses

| Course | Where | Cost |
|--------|-------|------|
| CS231n: CNNs for Visual Recognition | Stanford (YouTube) | Free |
| Deep Learning Specialization | Coursera (DeepLearning.AI) | ~$50/month |
| Fast.ai Practical Deep Learning | fast.ai | Free |
| EECS 498/598: Deep Learning (Michigan) | YouTube | Free |
| 6.S191: MIT Deep Learning | YouTube | Free |
| CS224N: NLP (for multimodal) | Stanford YouTube | Free |

### __ Tools to Master

- **Training:** PyTorch, PyTorch Lightning, Hugging Face Transformers
- **Tracking:** Weights & Biases (WandB), MLflow
- **Data:** Roboflow, Label Studio, FiftyOne
- **Deployment:** FastAPI + Docker, ONNX Runtime, TensorRT
- **3D Vision:** Open3D, PyTorch3D, tiny-cuda-nn (NeRF)
- **Edge:** TFLite, CoreML, OpenVINO, NCNN

### _ Paper Lists to Follow

- **Daily ArXiv:** cs.CV section -- browse titles every morning
- **Papers With Code:** paperswithcode.com -- SOTA with reproducible code
- **Two Minute Papers** (YouTube) -- visual summaries of key papers
- **The Batch** (deeplearning.ai newsletter) -- weekly curated digest

---

## 18.6 Career Pathways in CV (2030 Edition)

```
COMPUTER VISION CAREER PATHS
____________________________________________________________

ML/CV Research Scientist
  Entry: PhD or very strong publications / ICCV/CVPR papers
  Scope: Invent new architectures, publish research
  Companies: Google DeepMind, Meta AI, OpenAI, Apple ML Research

CV Applied Scientist  
  Entry: MS + strong project portfolio + internship
  Scope: Apply SOTA research to product problems
  Companies: Tesla Autopilot, Amazon Rekognition, Microsoft Azure CV

CV Software Engineer (MLE)
  Entry: BS/MS + system-level projects + deployment experience
  Scope: Build production CV systems at scale
  Companies: Any tech company with CV products

Robotics Vision Engineer
  Entry: Strong 3D vision + ROS + real-time systems
  Scope: Perception for robot arms, drones, mobile robots
  Companies: Figure, Boston Dynamics, Agility, Waymo, Cruise

Medical Imaging AI Engineer
  Entry: CV skills + domain knowledge + regulatory awareness
  Scope: FDA-cleared AI diagnostics
  Companies: Paige, Tempus, Recursion, Butterfly Network

CV Startup Founder / Indie Hacker
  Entry: Domain expertise + business sense + portfolio
  Scope: Build niche CV SaaS (inspection, agriculture, retail)
  Path: Build Projects F, D, A => launch on YC, Product Hunt

____________________________________________________________
Salary Range (2024 USD, scales upward to 2030):
  Entry MLE:         $130K-$180K
  Senior MLE:        $200K-$350K  
  Research Scientist: $250K-$500K+
  Staff/Principal:    $400K-$800K+
```

---

## 18.7 The 90-Day Intensive Plan (Zero to Portfolio)

```
Days 01-15: Foundation Mastery
  _ Complete Sections 1-5 of this notebook
  _ Run every code cell, modify and re-run
  _ Daily: build one small OpenCV script from scratch

Days 16-35: Deep Learning Core
  _ Complete Sections 6-9
  _ Train your first MNIST CNN to >99% accuracy
  _ Experiment with different architectures (LeNet, VGG, ResNet)
  _ Run CS231n Assignment 2

Days 36-55: Advanced Techniques
  _ Complete Sections 10-13
  _ Run YOLOv8 on a webcam / video file
  _ Fine-tune a model on custom dataset (Roboflow is helpful)
  _ Implement Grad-CAM from scratch

Days 56-75: Portfolio Projects
  _ Pick 3 projects from Section 18 list
  _ Implement Project A (VQA) -- multimodal is the future
  _ Implement Project D (Federated) -- ethics is 2030 signal
  _ Implement Project F (Synthetic data) -- shows systems thinking
  _ Push all code to GitHub with detailed READMEs

Days 76-90: Job-Ready Polish
  _ Deploy one project as a live demo (HuggingFace Spaces / Gradio)
  _ Write 3 LinkedIn posts explaining your projects technically
  _ Practice 20 interview questions from Section 16
  _ Apply to 10 positions with tailored cover letters
```

---

## 18.8 Final Cheat Sheet

```python
# __ IMAGE OPERATIONS __________________________________________________
import cv2, numpy as np, torch

img = cv2.imread('img.jpg')                        # load (BGR)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)     # BGR => RGB
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)       # RGB => grayscale
resized = cv2.resize(img, (w, h))                   # resize
crop = img[y1:y2, x1:x2]                           # crop
blurred = cv2.GaussianBlur(img, (5,5), 0)          # blur
edges = cv2.Canny(gray, 50, 150)                   # edge detection
_, thresh = cv2.threshold(gray, 127, 255, 0)       # threshold

# __ PYTORCH QUICK TRAINING ____________________________________________
model = MyModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
    model.train()
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
    
    model.eval()
    with torch.no_grad():
        # evaluate on val_loader

# __ TRANSFER LEARNING ________________________________________________
model = torchvision.models.resnet50(weights='IMAGENET1K_V2')
for p in model.parameters(): p.requires_grad = False   # freeze
model.fc = nn.Linear(2048, num_classes)                # new head
# only model.fc trains -- much faster!

# __ EVALUATION METRICS _______________________________________________
from sklearn.metrics import classification_report, confusion_matrix
iou = inter / (area1 + area2 - inter)                 # bounding box IoU
mAP = np.mean([ap_per_class(c) for c in classes])     # detection mAP

# __ DATA AUGMENTATION ________________________________________________
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.3, 0.3, 0.2),
    transforms.RandomErasing(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# __ GRAD-CAM ________________________________________________________
# Hook into last conv layer, get gradient w.r.t. target class score
# weights = global_avg_pool(gradients)  => cam = ReLU(sum(weights * activations))

# __ YOLO INFERENCE (ultralytics) _____________________________________
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
results = model('image.jpg')
for r in results: print(r.boxes.xyxy, r.boxes.conf, r.boxes.cls)
```

---

## 18.9 Congratulations -- You've Completed the Course! _

<div style="background: linear-gradient(135deg, #0f0c29, #302b63, #24243e); padding: 40px; border-radius: 15px; text-align: center; color: white; margin: 20px 0;">

<h2 style="font-size: 2em; margin-bottom: 15px;">_ Computer Vision Mastery Complete</h2>

<p style="font-size: 1.1em; color: #a8edea; margin-bottom: 25px;">
You've gone from zero to frontier-level Computer Vision knowledge.
</p>

<div style="display: flex; justify-content: center; gap: 30px; flex-wrap: wrap; margin: 20px 0;">
<div style="background: rgba(255,255,255,0.1); padding: 15px 25px; border-radius: 10px;">
<div style="font-size: 2em;">__</div>
<div>Image Processing</div>
<div style="color: #a8edea; font-size: 0.85em;">Mastered</div>
</div>
<div style="background: rgba(255,255,255,0.1); padding: 15px 25px; border-radius: 10px;">
<div style="font-size: 2em;">_</div>
<div>Deep Learning</div>
<div style="color: #a8edea; font-size: 0.85em;">Mastered</div>
</div>
<div style="background: rgba(255,255,255,0.1); padding: 15px 25px; border-radius: 10px;">
<div style="font-size: 2em;">_</div>
<div>Modern CV</div>
<div style="color: #a8edea; font-size: 0.85em;">Mastered</div>
</div>
<div style="background: rgba(255,255,255,0.1); padding: 15px 25px; border-radius: 10px;">
<div style="font-size: 2em;">_</div>
<div>2030 Projects</div>
<div style="color: #a8edea; font-size: 0.85em;">Ready to Build</div>
</div>
</div>

<p style="font-size: 0.95em; color: #ced4da; margin-top: 20px;">
The field moves fast. The best engineers read 2-3 papers per week.<br>
Your journey doesn't end here -- it accelerates.
</p>

</div>

---

> *"The best time to start learning Computer Vision was 10 years ago. The second best time is right now."*  
> -- Start building. The 2030 industry is waiting for people who understand this material deeply.
